# Microsoft Agent Framework — Azure OpenAI（Responses API）

在此程式碼範例中，您將使用 **Microsoft Agent Framework (MAF)** 來建立一個由 **Azure OpenAI** 支援的簡單代理，使用 **Responses API**。

> **遷移說明：** 此範例先前使用 Semantic Kernel 配合 GitHub Models。現已遷移至 Microsoft Agent Framework，且 GitHub Models（已棄用，將於 2026 年 7 月退休）已被取代為支援 Responses API 的 Azure OpenAI。MAF 中的 `OpenAIChatClient` 針對 Azure OpenAI 穩定的 `/openai/v1/` 端點，預設使用 Responses API。

此範例的目的是示範在實作各種代理模式時，稍後其他程式碼範例將會應用的步驟。


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## 匯入所需的 Python 套件


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## 定義工具

在 Microsoft Agent Framework 中，<strong>工具</strong> 是一個使用 `@tool` 裝飾的普通 Python 函數，代理可以調用它。以下我們定義一個工具，返回一個隨機的假期目的地，並避免重複前一個地點。


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## 建立代理程式

在這裡，我們建立名為 `TravelAgent` 的代理程式。

在此範例中，我們使用非常基本的指示。歡迎隨意修改這些指示，以觀察代理程式行為的變化。


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## 運行代理

現在我們可以運行代理。我們建立一個 `AgentSession`，讓代理能夠記住多輪對話，然後發送兩個 `user_inputs`。第一個是詢問一個行程；第二個則表示使用者不喜歡建議並請求另一個 —— 代理會利用會話歷史記錄以及 `get_random_destination` 工具來回應。

你可以修改這些訊息，以觀察代理不同的反應。回應會<strong>逐字元串流</strong>。


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
